# DVA HW4 Q2

Do not distribute or publish this code

Do not change the **#export** statements or add and other code or comments above them. They are needed for grading.

In [42]:
#export
'''
*** Imports ***
    DO NOT EDIT or add anything to this section
'''
import csv
import numpy as np  # http://www.numpy.org
import ast
from datetime import datetime
from math import log, floor, ceil
import random

## Update 'Utility' class

Modify the Utility class's methods. You can also add additional methods as required but don't change existing methods' arguments.

In [46]:
#export
class Utility(object):
    # This method computes entropy for information gain
    def entropy(self, class_y):
        # Input:
        #   class_y         : list of class labels (0's and 1's)

        # TODO: Compute the entropy for a list of classes
        #
        # Example:
        #    entropy([0,0,0,1,1,1,1,1,1]) = 0.9182958340544896
        # Do not round your result

        entropy = 0
        ### Implement your code here
        #############################################
        if not class_y:
            return 0
            
        total = len(class_y)
        count_1 = sum(class_y)
        count_0 = total - count_1
        
        p0 = count_0 / total if count_0 > 0 else 0
        p1 = count_1 / total if count_1 > 0 else 0
        
        if p0 > 0:
            entropy -= p0 * log(p0, 2)
        if p1 > 0:
            entropy -= p1 * log(p1, 2)
        #############################################
        return entropy

    def partition_classes(self, X, y, split_attribute, split_val):
        # Inputs:
        #   X               : data containing all attributes
        #   y               : labels
        #   split_attribute : column index of the attribute to split on
        #   split_val       : a numerical value to divide the split_attribute

        # TODO: Partition the data(X) and labels(y) based on the split value - BINARY SPLIT.
        #
        # Split_val should be a numerical value
        # For example, your split_val could be the mean of the values of split_attribute
        #
        # You can perform the partition in the following way
        # Numeric Split Attribute:
        #   Split the data X into two lists(X_left and X_right) where the first list has all
        #   the rows where the split attribute is less than or equal to the split value, and the
        #   second list has all the rows where the split attribute is greater than the split
        #   value. Also create two lists(y_left and y_right) with the corresponding y labels.

        X_left = []
        X_right = []
        y_left = []
        y_right = []
        ### Implement your code here
        #############################################
        for i in range(len(X)):
            if X[i][split_attribute] <= split_val:
                X_left.append(X[i])
                y_left.append(y[i])
            else:
                X_right.append(X[i])
                y_right.append(y[i])
        #############################################
        return (X_left, X_right, y_left, y_right)

    def information_gain(self, previous_y, current_y):
        # Inputs:
        #   previous_y: the distribution of original labels (0's and 1's)
        #   current_y:  the distribution of labels after splitting based on a particular
        #               split attribute and split value

        # TODO: Compute and return the information gain from partitioning the previous_y labels
        # into the current_y labels.
        # You will need to use the entropy function above to compute information gain
        # Reference: http://www.cs.cmu.edu/afs/cs.cmu.edu/academic/class/15381-s06/www/DTs.pdf

        info_gain = 0
        ### Implement your code here
        #############################################
        parent_entropy = self.entropy(previous_y)
        
        total = len(previous_y)
        weighted_child_entropy = 0
        
        for child in current_y:
            if child:
                weight = len(child) / total
                weighted_child_entropy += weight * self.entropy(child)
        
        info_gain = parent_entropy - weighted_child_entropy
        #############################################
        return info_gain

    def best_split(self, X, y):
        # Inputs:
        #   X       : Data containing all attributes
        #   y       : labels
        #   TODO    : For each node find the best split criteria and return the split attribute,
        #             spliting value along with  X_left, X_right, y_left, y_right (using partition_classes)
        #             in the dictionary format {'split_attribute':split_attribute, 'split_val':split_val,
        #             'X_left':X_left, 'X_right':X_right, 'y_left':y_left, 'y_right':y_right, 'info_gain':info_gain}

        split_attribute = 0
        split_val = 0
        X_left, X_right, y_left, y_right = [], [], [], []
        ### Implement your code here
        #############################################
        best_gain = -float('inf')
        n_features = len(X[0])
        
        # Randomly select sqrt(n_features) features
        n_features_to_try = max(1, int((n_features**0.5)))
        feature_indices = random.sample(range(n_features), n_features_to_try)
        
        for attribute in feature_indices:
            # Get unique values for the attribute
            values = sorted(set(x[attribute] for x in X))
            
            # Try each value as a split point
            for i in range(len(values)-1):
                split_val_candidate = (values[i] + values[i+1]) / 2
                
                # Partition data based on split value
                X_left_temp, X_right_temp, y_left_temp, y_right_temp = (
                    self.partition_classes(X, y, attribute, split_val_candidate))
                
                if len(y_left_temp) == 0 or len(y_right_temp) == 0:
                    continue
                    
                # Calculate information gain
                gain = self.information_gain(y, [y_left_temp, y_right_temp])
                
                # Update best split if this split is better
                if gain > best_gain:
                    best_gain = gain
                    split_attribute = attribute
                    split_val = split_val_candidate
                    X_left, X_right = X_left_temp, X_right_temp
                    y_left, y_right = y_left_temp, y_right_temp
        
        #############################################
        return {
            'split_attribute': split_attribute,
            'split_val': split_val,
            'X_left': X_left,
            'X_right': X_right,
            'y_left': y_left,
            'y_right': y_right,
            'info_gain': best_gain
        }
        #############################################
        return best_split
        pass
        #############################################

## Update the 'DecisionTree' and 'RandomForest' classes

Please modify the 'DecisionTree' and 'RandomForest' classes below

In [48]:
#export
class DecisionTree(object):
    def __init__(self, max_depth):
        # Initializing the tree as an empty dictionary or list, as preferred
        self.tree = {}
        self.max_depth = max_depth

    def learn(self, X, y, par_node = {}, depth=0):
        # TODO: Train the decision tree (self.tree) using the the sample X and labels y
        # You will have to make use of the functions in Utility class to train the tree

        # par_node is a parameter that is useful to pass additional information to call
        # the learn method recursively. Its not mandatory to use this parameter

        # Use the function best_split in Utility class to get the best split and
        # data corresponding to left and right child nodes

        # One possible way of implementing the tree:
        #    Each node in self.tree could be in the form of a dictionary:
        #       https://docs.python.org/2/library/stdtypes.html#mapping-types-dict
        #    For example, a non-leaf node with two children can have a 'left' key and  a
        #    'right' key. You can add more keys which might help in classification
        #    (eg. split attribute and split value)
        
        ### Implement your code here
        #############################################
        # Check stopping conditions
        if len(y) == 0:
            return
            
        # Check if all examples have same label
        if len(set(y)) == 1:
            self.tree = {'val': y[0]}
            return
            
        # Check max depth
        if depth >= self.max_depth:
            majority_vote = max(set(y), key=y.count)
            self.tree = {'val': majority_vote}
            return
            
        # Find best split
        utility = Utility()
        best_split = utility.best_split(X, y)
        
        if best_split['info_gain'] == -float('inf'):
            majority_vote = max(set(y), key=y.count)
            self.tree = {'val': majority_vote}
            return
            
        # Create internal node
        self.tree = {
            'split_attribute': best_split['split_attribute'],
            'split_val': best_split['split_val'],
            'left': {},
            'right': {}
        }
        
        # Recursively build subtrees
        left_tree = DecisionTree(self.max_depth)
        left_tree.learn(best_split['X_left'], best_split['y_left'], {}, depth + 1)
        self.tree['left'] = left_tree.tree
        
        right_tree = DecisionTree(self.max_depth)
        right_tree.learn(best_split['X_right'], best_split['y_right'], {}, depth + 1)
        self.tree['right'] = right_tree.tree
        #############################################

    def classify(self, record):
        # TODO: classify the record using self.tree and return the predicted label
        ### Implement your code here
        #############################################
        node = self.tree
        while 'val' not in node:
            if record[node['split_attribute']] <= node['split_val']:
                node = node['left']
            else:
                node = node['right']
        return node['val']
        #############################################

        #############################################
        #############################################
        pass 
        #############################################
        #############################################

In [50]:
#export
# This starter code does not run. You will have to add your changes and
# turn in code that runs properly.

"""
Here,
1. X is assumed to be a matrix with n rows and d columns where n is the
number of total records and d is the number of features of each record.
2. y is assumed to be a vector of labels of length n.
3. XX is similar to X, except that XX also contains the data label for each
record.
"""

"""
This skeleton is provided to help you implement the assignment.You must
implement the existing functions as necessary. You may add new functions
as long as they are called from within the given classes.

VERY IMPORTANT!
Do NOT change the signature of the given functions.
Do NOT change any part of the main function APART from the forest_size parameter.
"""
#export
class RandomForest(object):
    num_trees = 0
    decision_trees = []
    bootstraps_datasets = []
    bootstraps_labels = []

    def __init__(self, num_trees):
        # Initialization done here
        self.num_trees = num_trees
        self.decision_trees = [DecisionTree(max_depth=10) for i in range(num_trees)]
        self.bootstraps_datasets = []
        self.bootstraps_labels = []

    def _bootstrapping(self, XX, n):
        # Reference: https://en.wikipedia.org/wiki/Bootstrapping_(statistics)
        #
        # TODO: Create a sample dataset of size n by sampling with replacement
        #       from the original dataset XX.
        # Note that you would also need to record the corresponding class labels
        # for the sampled records for training purposes.

        sample = [] # sampled dataset
        labels = []  # class labels for the sampled records
        ### Implement your code here
        #############################################
        for _ in range(n):
            idx = random.randrange(len(XX))
            sample.append(XX[idx][:-1])  # All but last column
            labels.append(XX[idx][-1])   # Just last column
        #############################################
        return (sample, labels)

    def bootstrapping(self, XX):
        # Initializing the bootstap datasets for each tree
        for i in range(self.num_trees):
            data_sample, data_label = self._bootstrapping(XX, len(XX))
            self.bootstraps_datasets.append(data_sample)
            self.bootstraps_labels.append(data_label)

    def fitting(self):
        # TODO: Train `num_trees` decision trees using the bootstraps datasets
        # and labels by calling the learn function from your DecisionTree class.
        ### Implement your code here
        #############################################
        for i in range(self.num_trees):
            self.decision_trees[i].learn(
                self.bootstraps_datasets[i],
                self.bootstraps_labels[i]
            )
        #############################################

    def voting(self, X):
        y = []

        for record in X:
            # Following steps have been performed here:
            #   1. Find the set of trees that consider the record as an
            #      out-of-bag sample.
            #   2. Predict the label using each of the above found trees.
            #   3. Use majority vote to find the final label for this recod.
            votes = []

            for i in range(len(self.bootstraps_datasets)):
                dataset = self.bootstraps_datasets[i]

                if record not in dataset:
                    OOB_tree = self.decision_trees[i]
                    effective_vote = OOB_tree.classify(record)
                    votes.append(effective_vote)

            counts = np.bincount(votes)

            if len(counts) == 0:
                # TODO: Special case
                #  Handle the case where the record is not an out-of-bag sample
                #  for any of the trees.
                ### Implement your code here
                #############################################
                votes = [tree.classify(record) for tree in self.decision_trees]
                counts = np.bincount(votes)
                #############################################
            
            y.append(np.argmax(counts))

        return y
    def user(self):
        """
        :return: string
        your GTUsername, NOT your 9-Digit GTId
        """
        ### Implement your code here
        #############################################
        return 'nshukla46'
        #############################################


In [52]:
#export

# TODO: Determine the forest size according to your implementation.
# This function will be used by the autograder to set your forest size during testing
# VERY IMPORTANT: Minimum forest_size should be 10
def get_forest_size():
    forest_size = 30
    return forest_size

# TODO: Determine random seed to set for reproducibility
# This function will be used by the autograder to set the random seed to obtain the same results you achieve locally
def get_random_seed():
    random_seed = 42
    return random_seed

## Do not modify the below cell
The cell below is provided to test that your random forest classifier can be successfully built and run. Similar steps will be used to build and run your code in Gradescope. Any additional testing of functions can be done in the cells below the `run()` cell, as these will not be parsed by the autograder.

In [38]:
def run():
    np.random.seed(get_random_seed())
    # start time
    start = datetime.now()
    X = list()
    y = list()
    XX = list()  # Contains data features and data labels
    numerical_cols = set([i for i in range(0, 31)])  # indices of numeric attributes (columns)

    # Loading data set
    print("reading the data")
    with open("Wisconsin_breast_prognostic.csv") as f:
        next(f, None)
        for line in csv.reader(f, delimiter=","):
            xline = []
            for i in range(len(line)):
                if i in numerical_cols:
                    xline.append(ast.literal_eval(line[i]))
                else:
                    xline.append(line[i])

            X.append(xline[:-1])
            y.append(xline[-1])
            XX.append(xline[:])

    # Initializing a random forest.
    randomForest = RandomForest(get_forest_size())

    # printing the name
    print("__Name: " + randomForest.user()+"__")

    # Creating the bootstrapping datasets
    print("creating the bootstrap datasets")
    randomForest.bootstrapping(XX)

    # Building trees in the forest
    print("fitting the forest")
    randomForest.fitting()

    # Calculating an unbiased error estimation of the random forest
    # based on out-of-bag (OOB) error estimate.
    y_predicted = randomForest.voting(X)

    # Comparing predicted and true labels
    results = [prediction == truth for prediction, truth in zip(y_predicted, y)]

    # Accuracy
    accuracy = float(results.count(True)) / float(len(results))

    print("accuracy: %.4f" % accuracy)
    print("OOB estimate: %.4f" % (1 - accuracy))

    # end time
    print("Execution time: " + str(datetime.now() - start))

In [40]:
# Call the run() function to test your implementation
# Use this cell and any cells below for additional testing
run()

reading the data
__Name: nshukla46__
creating the bootstrap datasets
fitting the forest


NameError: name 'sqrt' is not defined